In [ ]:
!pip -q install -U "transformers>=4.43.0" accelerate pandas tqdm huggingface_hub

In [ ]:
import re
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
from huggingface_hub import login
login()

In [ ]:
MODEL_ID = "meta-llama/Llama-Guard-3-1B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
).eval()

PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

In [ ]:
@torch.inference_mode()
def _generate_guard_output(conversation, max_new_tokens: int = 32) -> str:
    enc = tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    prompt_len = enc["input_ids"].shape[1]

    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=PAD_TOKEN_ID,
        use_cache=True,
    )

    gen = out[:, prompt_len:]
    return tokenizer.decode(gen[0], skip_special_tokens=True).strip()


def llama_guard_prompt_only(prompt: str, max_new_tokens: int = 32) -> str:
    conversation = [{"role": "user", "content": [{"type": "text", "text": prompt or ""}]}]
    return _generate_guard_output(conversation, max_new_tokens=max_new_tokens)


def llama_guard_response_in_context(prompt: str, response: str, max_new_tokens: int = 32) -> str:
    conversation = [
        {"role": "user", "content": [{"type": "text", "text": prompt or ""}]},
        {"role": "assistant", "content": [{"type": "text", "text": response or ""}]},
    ]
    return _generate_guard_output(conversation, max_new_tokens=max_new_tokens)


def extract_safe_unsafe(raw: str) -> str:
    if not raw:
        return "unknown"
    m = re.search(r"\b(safe|unsafe)\b", raw.strip().lower())
    return m.group(1) if m else "unknown"

In [ ]:
INPUT_CSV = ""              
OUTPUT_CSV = ""   

df = pd.read_csv(INPUT_CSV)
required = {"prompt_id", "language", "prompt", "LLM_response"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}.")

combined = []

for i in tqdm(range(len(df)), desc="Llama-Guard-3-1B"):
    pid = df.at[i, "prompt_id"]
    lang = df.at[i, "language"]
    prompt = str(df.at[i, "prompt"])
    resp = str(df.at[i, "LLM_response"])

    raw_prompt = llama_guard_prompt_only(prompt, max_new_tokens=32)
    raw_resp = llama_guard_response_in_context(prompt, resp, max_new_tokens=32)

    prompt_label = extract_safe_unsafe(raw_prompt)
    resp_label = extract_safe_unsafe(raw_resp)

    combined_str = f"prompt_validation: {prompt_label}; response_validation: {resp_label}"
    print(f"[prompt_id={pid} lang={lang}] {combined_str}")

    combined.append(combined_str)

out_df = pd.DataFrame({
    "prompt_id": df["prompt_id"],
    "language": df["language"],
    "meta-llama/Llama-Guard-3-1B_validation": combined,
})
out_df.to_csv(OUTPUT_CSV, index=False)
print(f"Done: {OUTPUT_CSV}")
out_df.head()